# BairroVivo Vision — detecção em tempo real de problemas urbanos

**Componente de Visão do BairroVivo.** Um YOLO11n treinado no dataset público **CIVIC ISSUES v3** (Roboflow Universe,
CC BY 4.0, 43.584 imagens, 9 classes) detecta problemas de zeladoria — buraco, trinca, lixo, pichação, poste, placa e
concreto danificados, árvore caída — nas fotos de ocorrências enviadas por moradores. Cada detecção vira uma **sugestão** de categoria e prioridade para o
operador do BairroVivo confirmar.

| Etapa | O que faz |
|---|---|
| 1 | Configuração |
| 2 | Análise do dataset: classes, quantidade, exemplos |
| 3 | Vazamento: cópias da mesma foto em treino e teste (comum em exports do Roboflow) |
| 4 | Dataset final: novo split **por foto de origem**, classes escolhidas, tamanho compatível com a GPU |
| 5 | Treino do YOLO11n (transfer learning, pesos COCO) |
| 6 | Avaliação no teste: mAP, precisão, recall, F1, matriz de confusão e FPS |
| 7 | Integração: `analisar_imagem()` devolve a sugestão de ocorrência em JSON |
| 8 | Demo em tempo real: fotos de moradores (simuladas com o teste) chegando ao BairroVivo |
| 9 | App de fotos: roda o modelo nas 56 fotos de `fotos_a_serem_testadas/` e confere com o gabarito |

## 1. Configuração

In [2]:
import sys, os, json, shutil, subprocess, random, time
from pathlib import Path
from collections import Counter
import numpy as np, cv2

BASE = Path.cwd()                                   # D:\BairroVivo_Vision
sys.path.insert(0, str(BASE))
from vision_lib import *

ORIGEM = Path(r'D:\civic_issue_dataset')           # export do Roboflow (CIVIC ISSUES v3, formato YOLOv11)
ANALISE = BASE / 'analise'
DATASET = BASE / 'dataset' / 'final'
RUNS = BASE / 'runs'
MODELOS = BASE / 'modelos'
VIDEOS = BASE / 'videos'                            # vídeo da demo (montado na seção 8)

CLASSES_MANTIDAS = None      # None = decidir na seção 2 (ficam as classes com pelo menos MIN_IMAGENS imagens)
MIN_IMAGENS = 300
MAX_POR_SPLIT = {'train': 6000, 'val': 900, 'test': 900}   # cabe em ~1–2 h de treino numa RTX 3050
EPOCAS, PACIENCIA, IMGSZ, BATCH = 40, 10, 640, 16
HORAS_TREINO = 3.0         # limite de tempo: o Ultralytics ajusta o nº de épocas para caber nisso
SEED = 42
try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'
for d in (ANALISE, RUNS, MODELOS, VIDEOS):
    d.mkdir(parents=True, exist_ok=True)
DATA_YAML = achar_data_yaml(ORIGEM)
NOMES, SPLITS = ler_dataset(DATA_YAML)
print('data.yaml:', DATA_YAML)
print('classes:', NOMES)
print({sp: len(v) for sp, v in SPLITS.items()}, '· dispositivo:', DEVICE)

data.yaml: D:\civic_issue_dataset\data.yaml
classes: ['Damaged_Concrete_Structures', 'Damaged_Electric_Poles', 'Damaged_Road_Signs', 'Dead_Animal_Pollution', 'Fallen_Trees', 'Garbage', 'Graffiti', 'pothole', 'road_crack']
{'train': 34980, 'val': 4815, 'test': 3787} · dispositivo: 0


## 2. Análise do dataset

In [ ]:
ST = estatisticas(NOMES, SPLITS)
json.dump(ST, open(ANALISE / 'estatisticas_original.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
total = Counter()
for sp, s in ST.items():
    print(f"\n{sp}: {s['imagens']} imagens ({s['sem_rotulo']} sem rótulo)")
    for c, v in s['por_classe'].items():
        print(f"   {c:<36} {v['imagens']:>6} imagens {v['instancias']:>7} objetos")
        total[c] += v['imagens']
if CLASSES_MANTIDAS is None:
    CLASSES_MANTIDAS = [c for c in NOMES if total[c] >= MIN_IMAGENS]
print('\nClasses mantidas:', CLASSES_MANTIDAS)
print('Categoria no BairroVivo:')
for c in CLASSES_MANTIDAS:
    k = categoria(c); print(f"   {c:<36} → {k['categoria']} (prioridade {k['prioridade']})")
# tamanhos de imagem (amostra)
tam = Counter()
for img, _ in random.Random(0).sample(SPLITS['train'], min(200, len(SPLITS['train']))):
    im = imread(img); tam[im.shape[:2] if im is not None else None] += 1
print('Tamanhos mais comuns (amostra):', tam.most_common(3))
for i, c in enumerate(NOMES):
    if c in CLASSES_MANTIDAS:
        g = grade_exemplos(SPLITS['train'], NOMES, i, n=12)
        if g is not None:
            imwrite(ANALISE / f'exemplos_{c.replace(" ", "_")}.jpg', g, quality=85)
print('Mosaicos de exemplo em', ANALISE)

## 3. Vazamento entre treino, validação e teste
O Roboflow gera cópias aumentadas da mesma foto (`foto_jpg.rf.<hash>.jpg`). Se cópias da mesma foto caem em splits
diferentes, o teste vira "decoreba" e o mAP fica inflado. Verificamos pelo nome de origem e pelo *hash* perceptual.

In [ ]:
grupos, cruzados = vazamento_por_nome(SPLITS)
print(f'{len(grupos)} fotos de origem para {sum(len(v) for v in SPLITS.values())} imagens '
      f'({sum(len(v) for v in SPLITS.values()) / max(1, len(grupos)):.1f} cópias por foto)')
print(f'{len(cruzados)} fotos de origem aparecem em mais de um split (pelo nome)')
vh = vazamento_por_hash(SPLITS, max_dist=3, amostra=1500)
print('Pelo hash (amostra):', vh)
json.dump({'fotos_origem': len(grupos), 'cruzadas_por_nome': len(cruzados), 'hash': vh},
          open(ANALISE / 'vazamento.json', 'w', encoding='utf-8'), indent=2)

## 4. Dataset final (sem vazamento)
Agrupa as cópias de cada foto de origem, usa **uma** imagem por grupo e sorteia 80/10/10 **por grupo**. Mantém só as
classes escolhidas e limita o tamanho para o treino caber na RTX 3050 (amostragem que favorece as classes raras).

In [ ]:
import importlib, vision_lib; importlib.reload(vision_lib); from vision_lib import *
STF = montar_dataset(NOMES, SPLITS, DATASET, CLASSES_MANTIDAS, MAX_POR_SPLIT, seed=SEED)
json.dump(STF, open(ANALISE / 'estatisticas_final.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
for sp, s in STF.items():
    print(f"{sp:<6} {s['imagens']:>5} imagens ({s['negativas']} sem objeto) ·", s['instancias'])

## 5. Treino do YOLO11n (roda em outro processo; log em `runs/treino_vision.log`)
O disco D: lê só ~5 MB/s e o treino ficava preso na leitura das imagens (~14 min por época). Por isso o dataset final
(~400 MB) é copiado para o disco C: antes do treino.

In [ ]:
import gc; gc.collect()
NOME = 'bairrovivo_vision'
RAPIDO = Path(os.path.expanduser('~')) / 'bairrovivo_vision_dataset'     # no disco C: (mais rápido)
if not (RAPIDO / 'data.yaml').exists():
    if RAPIDO.exists():
        shutil.rmtree(RAPIDO)
    shutil.copytree(DATASET, RAPIDO)
    import yaml
    cfg = yaml.safe_load(open(RAPIDO / 'data.yaml', encoding='utf-8')); cfg['path'] = str(RAPIDO)
    (RAPIDO / 'data.yaml').write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
for npy in RAPIDO.rglob('*.npy'):
    npy.unlink()
print('Dataset de treino em', RAPIDO)
pesos_ini = BASE / 'yolo11n.pt'
if not pesos_ini.exists():
    from huggingface_hub import hf_hub_download
    shutil.copy2(hf_hub_download('Ultralytics/YOLO11', 'yolo11n.pt'), pesos_ini)
script = f'''
from ultralytics import YOLO
if __name__ == '__main__':
    YOLO(r'{pesos_ini}').train(data=r'{RAPIDO / 'data.yaml'}', epochs={EPOCAS}, patience={PACIENCIA}, imgsz={IMGSZ},
        batch={BATCH}, device={DEVICE!r}, workers=2, project=r'{RUNS}', name='{NOME}', exist_ok=True, seed={SEED},
        cache=False, plots=True, time={HORAS_TREINO})
'''
arq = RUNS / 'treinar_vision.py'; arq.write_text(script, encoding='utf-8')
log = open(RUNS / 'treino_vision.log', 'w', encoding='utf-8')
proc = subprocess.Popen([sys.executable, '-u', str(arq)], cwd=str(BASE), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding='utf-8', errors='replace')
try:
    for linha in proc.stdout:
        log.write(linha); log.flush()
        linha = linha.rstrip().split('\r')[-1]
        if linha and ('all ' in linha or 'epochs completed' in linha or 'Error' in linha or 'EarlyStopping' in linha):
            print(linha[-170:], flush=True)
finally:
    if proc.poll() is None:
        proc.kill()
    log.close()
if proc.wait() != 0:
    raise RuntimeError(f'Treino falhou; veja {RUNS / "treino_vision.log"}')

## 6. Avaliação no teste

In [ ]:
from ultralytics import YOLO
import yaml
CLASSES_MANTIDAS = [v for k, v in sorted(yaml.safe_load(open(DATASET / 'data.yaml', encoding='utf-8'))['names'].items())]
MODELO = MODELOS / 'bairrovivo_vision.pt'
MELHOR = RUNS / 'bairrovivo_vision' / 'weights' / 'best.pt'   # existe logo após o treino da seção 5
if MELHOR.exists():
    shutil.copy2(MELHOR, MODELO)
m = YOLO(str(MODELO))
r = m.val(data=str(DATASET / 'data.yaml'), split='test', device=DEVICE, workers=0, plots=True,
          project=str(RUNS), name='bairrovivo_vision_teste', exist_ok=True)
res = {'map50': float(r.box.map50), 'map50_95': float(r.box.map), 'precisao': float(r.box.mp), 'recall': float(r.box.mr),
       'por_classe': {}}
for i, c in enumerate(CLASSES_MANTIDAS):
    try:
        p, rc = float(r.box.p[i]), float(r.box.r[i])
        res['por_classe'][c] = {'precisao': p, 'recall': rc, 'f1': 2 * p * rc / max(1e-9, p + rc), 'map50': float(r.box.ap50[i])}
    except IndexError:
        pass
# velocidade (FPS) na GPU
imgs = sorted((DATASET / 'images' / 'test').glob('*'))[:100]
m.predict(str(imgs[0]), verbose=False, device=DEVICE)
t0 = time.time()
for p in imgs:
    m.predict(str(p), verbose=False, device=DEVICE)
res['fps'] = len(imgs) / (time.time() - t0)
json.dump(res, open(MODELOS / 'metricas_vision.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print(f"Teste: mAP50 {res['map50']:.3f} · mAP50-95 {res['map50_95']:.3f} · P {res['precisao']:.3f} · R {res['recall']:.3f} · {res['fps']:.0f} FPS")
for c, v in res['por_classe'].items():
    print(f"   {c:<36} P={v['precisao']:.2f} R={v['recall']:.2f} F1={v['f1']:.2f} mAP50={v['map50']:.2f}")
print('Matriz de confusão e curvas em', RUNS / 'bairrovivo_vision_teste')

## 7. Integração com o BairroVivo: `analisar_imagem()`
É o que o backend chama quando o morador envia a foto. Devolve categoria, prioridade, confiança e as caixas; com
confiança baixa marca `revisao_manual = true`.

In [ ]:
import importlib, analisar_ocorrencia; importlib.reload(analisar_ocorrencia)
from analisar_ocorrencia import analisar_imagem, desenhar
exemplos = random.Random(1).sample(sorted((DATASET / 'images' / 'test').glob('*')), 4)
for p in exemplos:
    resp = analisar_imagem(p, MODELOS / 'bairrovivo_vision.pt', device=DEVICE)
    print(p.name, json.dumps(resp['sugestao'], ensure_ascii=False), '· revisão manual:', resp['revisao_manual'])
    imwrite(ANALISE / f'exemplo_api_{p.stem}.jpg', desenhar(imread(p), resp))

## 8. Demo em tempo real: fotos de moradores chegando ao BairroVivo
Uso principal do componente. Como não temos fotos reais do bairro, **simulamos** o envio com fotos **sorteadas** do
conjunto de teste (nunca vistas no treino), ~igual por classe, 2 s cada. O app detecta ao vivo e gera as ocorrências.

In [3]:
import importlib, montar_demo_fotos; importlib.reload(montar_demo_fotos)
DEMO = VIDEOS / 'demo_fotos_moradores.mp4'
fotos = montar_demo_fotos.montar(DATASET, DEMO, n=60, seg=2.0)
print(len(fotos), 'fotos no vídeo', DEMO)
cmd = [sys.executable, str(BASE / 'app_vision.py'), '--video', str(DEMO), '--modelo', str(MODELOS / 'bairrovivo_vision.pt'),
       '--sem-janela', '--salvar']
r = subprocess.run(cmd, cwd=str(BASE), capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout[-1500:]); print(r.stderr[-1500:]) if r.returncode else None

# confere cada foto com o rótulo real: categoria certa, nada detectado ou só categoria errada
import csv as _csv, yaml
nomes_cls = yaml.safe_load(open(DATASET / 'data.yaml', encoding='utf-8'))['names']
ocs = list(_csv.DictReader(open(BASE / 'saidas' / 'ocorrencias_demo_fotos_moradores.csv', encoding='utf-8')))
por_foto = {i: set() for i in range(len(fotos))}
for o in ocs:
    por_foto[min(len(fotos) - 1, int(float(o['t']) // 2.0))].add(o['classe'])
res = Counter()
for i, p in enumerate(fotos):
    real = {nomes_cls[int(l.split()[0])] for l in open(DATASET / 'labels' / 'test' / f'{p.stem}.txt') if l.strip()}
    res['nada detectado' if not por_foto[i] else 'categoria certa' if por_foto[i] & real else 'só categoria errada'] += 1
print(dict(res), f"· quando sugere, acerta {res['categoria certa']}/{res['categoria certa'] + res['só categoria errada']}")
json.dump({'fotos': len(fotos), 'ocorrencias': len(ocs), **res}, open(ANALISE / 'demo_resultado.json', 'w', encoding='utf-8'),
          indent=2, ensure_ascii=False)

56 fotos no vídeo d:\BairroVivo_Vision\videos\demo_fotos_moradores.mp4
52 ocorrências sugeridas: {'Iluminação / rede elétrica': 9, 'Buraco / pavimento': 13, 'Via obstruída': 7, 'Descarte irregular': 4, 'Calçada / acessibilidade': 8, 'Mobiliário urbano': 11}
Lista em D:\BairroVivo_Vision\saidas\ocorrencias_demo_fotos_moradores.csv

{'categoria certa': 35, 'nada detectado': 20, 'só categoria errada': 1} · quando sugere, acerta 35/36


## 9. App de fotos: `fotos_a_serem_testadas/`
As mesmas 56 fotos da demo da seção 8 (7 por classe, do conjunto de teste) ficam soltas na pasta
`fotos_a_serem_testadas/`, com o rótulo real em `gabarito.csv`. O `app_fotos.py` roda o modelo em cada uma, mostra a
sugestão de ocorrência e diz se acertou. Aqui roda sem janela; para ver as fotos uma a uma: `python app_fotos.py`.

In [ ]:
cmd = [sys.executable, str(BASE / 'app_fotos.py'), '--sem-janela']
r = subprocess.run(cmd, cwd=str(BASE), capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout[-2500:]); print(r.stderr[-1500:]) if r.returncode else None